In [1]:
import json
import matplotlib.pyplot as plt
import torch
import numpy as np
import random
import rich
import matplotlib.pyplot as plt

from rich.progress import Progress
from torchinfo import summary
from torch import nn
from torchvision import datasets, models
from torchvision.transforms import v2
from torch.utils.data import Subset, ConcatDataset, DataLoader

In [2]:
# Set random seed for reproducibility
RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)

In [3]:
with open("split_indices.json", "r") as f:
    indices = json.load(f)

# Get the indices for each split
train_indices = indices["train"]
val_indices = indices["val"]
test_indices = indices["test"]

# Check if number of indices in each split is consistent
print(f"Train: {len(train_indices)}\nVal: {len(val_indices)}\nTest: {len(test_indices)}")

Train: 2284
Val: 490
Test: 490


# DATA LOADING

In [ ]:
!unzip -q Dataset.zip

In [4]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms() # Transformation steps for ResNet50

train_transforms = v2.Compose([v2.ToImage(),
                               v2.ToDtype(dtype=torch.float32, scale=False),
                               v2.RandomHorizontalFlip(),
                               v2.RandomRotation(15),
                               v2.GaussianNoise(sigma=0.02),
                               preprocess])

test_transforms = v2.Compose([preprocess])

In [11]:
# Ignore the current train/test split. Combine into 1 unified dataset before split using indices for reproducibility
DATASET1_PATH = "/home/sagemaker-user/A3/Training"
DATASET2_PATH = "/home/sagemaker-user/A3/Testing"

# TRAIN DATASET + DATALOADER
train_dataset = Subset(ConcatDataset([datasets.ImageFolder(DATASET1_PATH, transform=train_transforms),
                                      datasets.ImageFolder(DATASET2_PATH, transform=train_transforms)]), 
                                      train_indices)
train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

# VALIDATION DATASET + DATALOADER (SAME TRANSFORMATION AS TEST DATASET)
val_dataset = Subset(ConcatDataset([datasets.ImageFolder(DATASET1_PATH, transform=test_transforms),
                                    datasets.ImageFolder(DATASET2_PATH, transform=test_transforms)]), 
                                    val_indices)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=16, shuffle=False)

# TEST DATASET
test_dataset = Subset(ConcatDataset([datasets.ImageFolder(DATASET1_PATH, transform=test_transforms),
                                     datasets.ImageFolder(DATASET2_PATH, transform=test_transforms)]), 
                                     test_indices)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=16, shuffle=False)

# MODEL TRAINING

In [6]:
# DOWNLOAD + MODIFY MODEL
model = models.resnet18(weights=weights, progress=True)
model.fc = nn.Sequential(nn.Dropout(p=0.6),
                         nn.Linear(in_features=512, out_features=4, bias=True))

# Unfreeze the classifier head
#for param in model.fc.parameters():
#    param.requires_grad = True

# Freeze the convolution backbone
#for name, param in model.named_parameters():
#    if "fc" not in name:
#        param.requires_grad = False

for param in model.parameters():
    param.requires_grad = True
    
summary(model, input_size=(32, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [32, 4]                   --
├─Conv2d: 1-1                            [32, 64, 112, 112]        9,408
├─BatchNorm2d: 1-2                       [32, 64, 112, 112]        128
├─ReLU: 1-3                              [32, 64, 112, 112]        --
├─MaxPool2d: 1-4                         [32, 64, 56, 56]          --
├─Sequential: 1-5                        [32, 64, 56, 56]          --
│    └─BasicBlock: 2-1                   [32, 64, 56, 56]          --
│    │    └─Conv2d: 3-1                  [32, 64, 56, 56]          36,864
│    │    └─BatchNorm2d: 3-2             [32, 64, 56, 56]          128
│    │    └─ReLU: 3-3                    [32, 64, 56, 56]          --
│    │    └─Conv2d: 3-4                  [32, 64, 56, 56]          36,864
│    │    └─BatchNorm2d: 3-5             [32, 64, 56, 56]          128
│    │    └─ReLU: 3-6                    [32, 64, 56, 56]          --
│

In [9]:
# SET UP HYPERPARAMETERS
LEARNING_RATE = 0.0001
EPOCHS = 100
PATIENCE = 10
NO_IMPROVE = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHT_DECAY = 0.001
BEST_VAL_LOSS = float("inf")

if WEIGHT_DECAY == None:
    OPTIMIZER = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
else:
    OPTIMIZER = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

SCHEDULER = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=OPTIMIZER, mode="min", factor=0.5, patience=PATIENCE/2)

# CALCULATE CLASS WEIGHTS
class_count = [0] * 4

for _, label in train_dataset:
    class_count[label] += 1

class_weights = torch.tensor(list(map(lambda x: 1 / x, class_count)), dtype=torch.float).to(DEVICE)
LOSS_FUNC = nn.CrossEntropyLoss(weight=class_weights)

In [10]:
# TRAINING LOOP
model.to(DEVICE)
train_loss_over_time = []
val_loss_over_time = []
    
with Progress() as progress:
    epoch_counter = progress.add_task(description="EPOCH", total=EPOCHS)
    batch_counter = progress.add_task(description="BATCH", total=len(train_dataloader))

    for epoch in range(EPOCHS):
        train_loss, train_correct = 0, 0
        val_loss, val_correct = 0, 0
        
        # ================================= TRAIN ===========================================
        for images, labels in train_dataloader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            model.train()
            y_logits = model(images)
            y_probs = torch.softmax(y_logits, dim=1)
            y_preds = torch.argmax(y_probs, dim=1)
            train_correct += (y_preds == labels).sum().item()
            
            loss = LOSS_FUNC(y_logits, labels)
            train_loss += loss.item()  # Accumlate loss of each batch
            OPTIMIZER.zero_grad()
            loss.backward()  # Backpropagation
            OPTIMIZER.step()

            progress.update(batch_counter, advance=1)

        train_loss = train_loss / len(train_dataloader)
        train_acc = train_correct / len(train_dataset)
        train_loss_over_time.append(train_loss) # Record loss for batch per epoch

        # ================================= VALIDATE ============================================
        model.eval()

        with torch.inference_mode():
            for images, labels in val_dataloader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
    
                y_logits = model(images)
                y_probs = torch.softmax(y_logits, dim=1)
                y_preds = torch.argmax(y_probs, dim=1)
                val_correct += (y_preds == labels).sum().item()
                
                loss = LOSS_FUNC(y_logits, labels)
                val_loss += loss.item()

        val_loss = val_loss / len(val_dataloader)
        val_acc = val_correct / len(val_dataset)
        val_loss_over_time.append(val_loss)
        progress.update(epoch_counter, advance=1)
        progress.reset(batch_counter)
        SCHEDULER.step(val_loss)
        print(f"TRAIN: Loss = {train_loss: .3f} Accuracy = {train_acc: .3f} | VAL: Loss = {val_loss: .3f} Accuracy = {val_acc: .3f}")
        
        # ===================== EARLY STOPPING ==================================
        if val_loss < BEST_VAL_LOSS:
            BEST_VAL_LOSS = val_loss
            NO_IMPROVE = 0
            torch.save(model.state_dict(), "best_weights.pth")
        else:
            NO_IMPROVE += 1
            if NO_IMPROVE >= PATIENCE:
                print("Early stopping triggered")
                break

Output()

TRAIN: Loss =  1.057 Accuracy =  0.545 | VAL: Loss =  1.922 Accuracy =  0.112

TRAIN: Loss =  0.812 Accuracy =  0.662 | VAL: Loss =  2.363 Accuracy =  0.153

TRAIN: Loss =  0.745 Accuracy =  0.694 | VAL: Loss =  1.994 Accuracy =  0.267

TRAIN: Loss =  0.660 Accuracy =  0.724 | VAL: Loss =  3.515 Accuracy =  0.153

...

TRAIN: Loss =  0.466 Accuracy =  0.815 | VAL: Loss =  2.791 Accuracy =  0.131

TRAIN: Loss =  0.429 Accuracy =  0.825 | VAL: Loss =  3.365 Accuracy =  0.135

TRAIN: Loss =  0.381 Accuracy =  0.843 | VAL: Loss =  3.026 Accuracy =  0.163

TRAIN: Loss =  0.379 Accuracy =  0.845 | VAL: Loss =  2.890 Accuracy =  0.224

Early stopping triggered

In [ ]:
# Save training and validation loss
loss_over_time = tuple(zip(train_loss_over_time, val_loss_over_time))

with open("loss_over_time.txt", "w") as f:
    for losses in loss_over_time:
        f.write(f"{losses[0]} {losses[1]}")

In [ ]:
# Display loss curve
train_loss_over_time = []
val_loss_over_time = []

with open("loss_over_time.txt", "r") as f:
    for line in f:
        line = line.split()
        train_loss_over_time.append(float(line[0]))
        val_loss_over_time.append(float(line[0]))

plt.figure(figsize=(10, 10))
plt.plot(range(len(train_loss_over_time)), train_loss_over_time)
plt.plot(range(len(val_loss_over_time)), val_loss_over_time)